# YOLOv8 Pruning + QAT Pipeline (Full Notebook)

Notebook nay chay full pipeline:

1. Sparsity training
2. Structured pruning
3. Finetune model da prune
4. QAT tren model pruned
5. Export QAT ONNX va build TensorRT engine (neu co `trtexec`)

Ban chi can sua cell `Config` ben duoi neu duong dan khac voi may cua ban.


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

import yaml
import torch

IN_COLAB = "google.colab" in sys.modules
ON_WINDOWS = os.name == "nt"

print(f"IN_COLAB   : {IN_COLAB}")
print(f"OS         : {os.name}")
print(f"Python     : {sys.version.split()[0]}")
print(f"Torch      : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")


In [ ]:
# Optional: mount Google Drive neu ban chay tren Colab
MOUNT_DRIVE = False

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted")
elif IN_COLAB:
    print("Dang o Colab nhung MOUNT_DRIVE=False")
else:
    print("Dang o local Jupyter, bo qua mount Drive")


In [ ]:
# =========================
# Config (sua neu can)
# =========================
cwd = Path.cwd().resolve()

if (cwd / "yolov8-pruning").exists():
    REPO = cwd / "yolov8-pruning"
elif (cwd / "prune.py").exists() and (cwd / "qat_pruned.py").exists():
    REPO = cwd
else:
    REPO = Path(r"C:\AI Project\YOLO\yolov8-pruning")

WORK_ROOT = REPO.parent

if IN_COLAB:
    # Sua path nay neu dataset tren Drive cua ban o noi khac
    DATASET_SOURCE = Path("/content/drive/MyDrive/yolov8/dataset")
    DATASET_LOCAL = Path("/content/dataset")
else:
    DATASET_SOURCE = WORK_ROOT / "dataset"
    DATASET_LOCAL = DATASET_SOURCE

PRETRAINED_CANDIDATES = [
    REPO / "weights" / "original.pt",
    WORK_ROOT / "phase_C.pt",
    WORK_ROOT / "phase_D.pt",
    WORK_ROOT / "phase_B.pt",
    WORK_ROOT / "phase_A.pt",
]
PRETRAINED_WEIGHT = next((p for p in PRETRAINED_CANDIDATES if p.exists()), PRETRAINED_CANDIDATES[0])

NC = 3
CLASS_NAMES = ["PAPER", "PLASTIC", "GLASS"]

MODEL_SIZE = "s"  # n, s, m, l, x
MODEL_YAML = f"yolov8{MODEL_SIZE}.yaml"
MODEL_CFG_PATH = REPO / "ultralytics" / "cfg" / "models" / "v8" / "yolov8.yaml"

RUNS_DIR = REPO / "runs"
WEIGHTS_DIR = REPO / "weights"
DATA_YAML_PATH = REPO / "data.pruned_qat.yaml"

DEVICE = 0
TRAIN_WORKERS = 0 if ON_WINDOWS else 8
QAT_WORKERS = 0 if ON_WINDOWS else 8
CACHE_MODE = "ram" if IN_COLAB else "disk"

ENGINE_PATH = None

print(f"REPO             : {REPO}")
print(f"DATASET_SOURCE   : {DATASET_SOURCE}")
print(f"DATASET_LOCAL    : {DATASET_LOCAL}")
print(f"PRETRAINED_WEIGHT: {PRETRAINED_WEIGHT}")
print(f"DEVICE           : {DEVICE}")


In [ ]:
def run_cmd(cmd, cwd=REPO, check=True):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=str(cwd), check=check)


def assert_exists(path, label="Path"):
    p = Path(path)
    assert p.exists(), f"{label} not found: {p}"
    return p


assert torch.cuda.is_available(), "GPU is required for pruning + QAT pipeline"

assert_exists(REPO / "prune.py", "prune.py")
assert_exists(REPO / "qat_pruned.py", "qat_pruned.py")
assert_exists(REPO / "qat_pruned_export.py", "qat_pruned_export.py")
assert_exists(MODEL_CFG_PATH, "yolov8.yaml")
assert_exists(DATASET_SOURCE, "Dataset source")
assert_exists(PRETRAINED_WEIGHT, "Pretrained weight")

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Sanity check OK")


In [ ]:
# Self-heal helpers neu chay cell le
from pathlib import Path


def _is_repo(path: Path) -> bool:
    return (
        (path / "pyproject.toml").exists()
        and (path / "prune.py").exists()
        and (path / "ultralytics").exists()
    )


if "REPO" not in globals() or not _is_repo(Path(REPO)):
    candidates = [
        Path.cwd().resolve(),
        Path("/content/yolov8-pruning"),
        Path("/content/drive/MyDrive/yolov8-pruning"),
        Path("/content/drive/MyDrive/YOLO/yolov8-pruning"),
        Path("/content/drive/MyDrive/QAT/yolov8-pruning"),
        Path("/content/drive/MyDrive/QAT/yolov8-QAT"),
    ]

    found = None
    for c in candidates:
        if _is_repo(c):
            found = c
            break

    # Quet nong them o 2 root pho bien tren Colab
    if found is None:
        for root in [Path("/content"), Path("/content/drive/MyDrive")]:
            if not root.exists():
                continue
            for hit in root.rglob("pyproject.toml"):
                cand = hit.parent
                if _is_repo(cand):
                    found = cand
                    break
            if found is not None:
                break

    if found is None:
        raise AssertionError(
            "Khong tim thay repo yolov8-pruning. Hay set REPO dung path, vd: "
            "REPO = Path('/content/drive/MyDrive/yolov8-pruning')"
        )

    REPO = found

if "run_cmd" not in globals():
    import subprocess

    def run_cmd(cmd, cwd=REPO, check=True):
        cmd = [str(x) for x in cmd]
        print("$", " ".join(cmd))
        return subprocess.run(cmd, cwd=str(cwd), check=check)

if "assert_exists" not in globals():

    def assert_exists(path, label="Path"):
        p = Path(path)
        assert p.exists(), f"{label} not found: {p}"
        return p

print("Using REPO:", REPO)

# Install dependencies
assert_exists(REPO / "pyproject.toml", "pyproject.toml")
assert_exists(REPO / "ultralytics", "ultralytics package dir")

# Khong dung -q o cell nay de de debug neu bi loi moi truong
run_cmd([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPO)
run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "setuptools", "wheel"], cwd=REPO)
run_cmd([sys.executable, "-m", "pip", "install", "nvidia-pyindex"], cwd=REPO)
run_cmd([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "pytorch-quantization==2.1.2",
    "--extra-index-url",
    "https://pypi.nvidia.com",
], cwd=REPO)
run_cmd([sys.executable, "-m", "pip", "install", "numpy<2.0"], cwd=REPO)

# Verify import trong chinh kernel hien tai
try:
    import ultralytics
except ModuleNotFoundError:
    # Fallback: import truc tiep tu source tree
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    import ultralytics

import pytorch_quantization

print("Python executable     :", sys.executable)
print("Ultralytics file      :", ultralytics.__file__)
print("Ultralytics version   :", ultralytics.__version__)
print("pytorch_quantization  :", pytorch_quantization.__version__)
print("Dependency setup done")


In [ ]:
# Prepare dataset + write data yaml
if IN_COLAB:
    if DATASET_LOCAL.exists():
        shutil.rmtree(DATASET_LOCAL)
    shutil.copytree(DATASET_SOURCE, DATASET_LOCAL)
    print(f"Copied dataset -> {DATASET_LOCAL}")

for split in ["train", "val", "test"]:
    img_dir = DATASET_LOCAL / split / "images"
    lbl_dir = DATASET_LOCAL / split / "labels"
    assert img_dir.exists(), f"Missing images dir: {img_dir}"
    assert lbl_dir.exists(), f"Missing labels dir: {lbl_dir}"
    print(f"{split:5s}: {len(list(img_dir.glob('*'))):5d} images")

data_yaml = {
    "train": str(DATASET_LOCAL / "train" / "images"),
    "val": str(DATASET_LOCAL / "val" / "images"),
    "test": str(DATASET_LOCAL / "test" / "images"),
    "nc": NC,
    "names": CLASS_NAMES,
}

with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("Wrote:", DATA_YAML_PATH)
print(DATA_YAML_PATH.read_text(encoding="utf-8"))


## Step 1 - Sparsity Training

Train model voi sparsity regularization de chuan bi cho pruning.


In [ ]:
SPARSITY_EPOCHS = 50
SPARSITY_BATCH = 32
SPARSITY_SR = 0.01
SPARSITY_LR0 = 1e-3
SPARSITY_PATIENCE = 50
IMGSZ = 640


In [ ]:
from ultralytics import YOLO

model = YOLO(str(PRETRAINED_WEIGHT))
model.train(
    data=str(DATA_YAML_PATH),
    epochs=SPARSITY_EPOCHS,
    imgsz=IMGSZ,
    batch=SPARSITY_BATCH,
    sr=SPARSITY_SR,
    lr0=SPARSITY_LR0,
    patience=SPARSITY_PATIENCE,
    workers=TRAIN_WORKERS,
    cache=CACHE_MODE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="train-sparsity",
    exist_ok=True,
)


In [ ]:
SPARSITY_WEIGHT = RUNS_DIR / "train-sparsity" / "weights" / "last.pt"
assert_exists(SPARSITY_WEIGHT, "Step1 output (last.pt)")
print("Step 1 done:", SPARSITY_WEIGHT)


## Step 2 - Structured Pruning

Cat tia cac channels/filter it quan trong dua tren BN gamma.


In [ ]:
PRUNE_RATIO = 0.05

run_cmd([
    sys.executable,
    "prune.py",
    "--weights", str(SPARSITY_WEIGHT),
    "--cfg", str(MODEL_CFG_PATH),
    "--model-size", MODEL_SIZE,
    "--prune-ratio", str(PRUNE_RATIO),
    "--save-dir", str(WEIGHTS_DIR),
], cwd=REPO)


In [ ]:
import glob

pruned_files = sorted(glob.glob(str(WEIGHTS_DIR / "pruned*.pt")))
assert pruned_files, f"No pruned weight found in: {WEIGHTS_DIR}"
PRUNED_WEIGHT = Path(pruned_files[-1])

ckpt = torch.load(PRUNED_WEIGHT, map_location="cpu", weights_only=False)
assert "maskbndict" in ckpt, "Invalid pruned checkpoint: missing maskbndict"

params = sum(p.numel() for p in ckpt["model"].parameters())
print("Step 2 done:", PRUNED_WEIGHT)
print("Pruned params:", f"{params:,}")
print("maskbndict keys:", len(ckpt["maskbndict"]))


## Step 3 - Finetune Pruned Model

Recovery finetune de phuc hoi accuracy sau pruning.


In [ ]:
FINETUNE_EPOCHS = 150
FINETUNE_BATCH = 32
FINETUNE_PATIENCE = 50

model = YOLO(str(PRUNED_WEIGHT))
model.train(
    data=str(DATA_YAML_PATH),
    epochs=FINETUNE_EPOCHS,
    imgsz=IMGSZ,
    batch=FINETUNE_BATCH,
    sr=0.0,
    patience=FINETUNE_PATIENCE,
    workers=TRAIN_WORKERS,
    cache=CACHE_MODE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="train-finetune",
    exist_ok=True,
)


In [ ]:
best_path = RUNS_DIR / "train-finetune" / "weights" / "best.pt"
last_path = RUNS_DIR / "train-finetune" / "weights" / "last.pt"

FINETUNE_WEIGHT = best_path if best_path.exists() else last_path
assert_exists(FINETUNE_WEIGHT, "Step3 output (best.pt/last.pt)")
print("Step 3 done:", FINETUNE_WEIGHT)


## Step 4 - QAT on Pruned Model

QAT voi fake INT8 quantization tren model da prune (+ finetune).


In [ ]:
QAT_EPOCHS = 10
QAT_BATCH = 16
QAT_LR0 = 1e-3  # trainer se tu scale lr0/100
QAT_FREEZE = 10
QAT_OPTIMIZER = "AdamW"
QAT_PATIENCE = 5
QAT_RECALIB_EVERY = 1

qat_cmd = [
    sys.executable,
    "qat_pruned.py",
    "--pruned-checkpoint", str(PRUNED_WEIGHT),
    "--pretrained-weight", str(FINETUNE_WEIGHT),
    "--model-config", MODEL_YAML,
    "--data-config", str(DATA_YAML_PATH),
    "--epochs", str(QAT_EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(QAT_BATCH),
    "--device", str(DEVICE),
    "--workers", str(QAT_WORKERS),
    "--cache", CACHE_MODE,
    "--lr0", str(QAT_LR0),
    "--freeze", str(QAT_FREEZE),
    "--optimizer", QAT_OPTIMIZER,
    "--patience", str(QAT_PATIENCE),
    "--recalib-every", str(QAT_RECALIB_EVERY),
    "--project", str(RUNS_DIR / "qat-pruned"),
    "--name", "train",
    "--exist-ok",
    "--plots",
]

run_cmd(qat_cmd, cwd=REPO)


In [ ]:
qat_best = RUNS_DIR / "qat-pruned" / "train" / "weights" / "best.pt"
qat_last = RUNS_DIR / "qat-pruned" / "train" / "weights" / "last.pt"

QAT_WEIGHT = qat_best if qat_best.exists() else qat_last
assert_exists(QAT_WEIGHT, "Step4 output (best.pt/last.pt)")
print("Step 4 done:", QAT_WEIGHT)


## Step 5 - Export QAT ONNX and Build TensorRT Engine

Export sang ONNX co Q/DQ nodes. Build `.engine` neu may co `trtexec`.


In [ ]:
run_cmd([
    sys.executable,
    "qat_pruned_export.py",
    "--pruned-checkpoint", str(PRUNED_WEIGHT),
    "--weight", str(QAT_WEIGHT),
], cwd=REPO)

onnx_candidates = sorted((QAT_WEIGHT.parent).glob("*.tensorrt.onnx"))
assert onnx_candidates, f"No .tensorrt.onnx found in {QAT_WEIGHT.parent}"
ONNX_PATH = onnx_candidates[-1]

print("Step 5a done:", ONNX_PATH)
print("ONNX size (MB):", round(ONNX_PATH.stat().st_size / 1024 / 1024, 2))


In [ ]:
trtexec_bin = shutil.which("trtexec")

if trtexec_bin is None:
    print("trtexec not found -> skip TensorRT build")
    ENGINE_PATH = None
else:
    ENGINE_PATH = ONNX_PATH.with_suffix(".engine")
    try:
        run_cmd([
            trtexec_bin,
            f"--onnx={ONNX_PATH}",
            f"--saveEngine={ENGINE_PATH}",
            "--int8",
            "--fp16",
            "--verbose",
        ], cwd=REPO)
    except subprocess.CalledProcessError as e:
        print("TensorRT build failed:", e)
        ENGINE_PATH = None

    if ENGINE_PATH is not None and Path(ENGINE_PATH).exists():
        print("Step 5b done:", ENGINE_PATH)
        print("Engine size (MB):", round(Path(ENGINE_PATH).stat().st_size / 1024 / 1024, 2))
    else:
        print("Engine not created")


## Optional - Quick Validation and Inference Preview

Cell duoi day la optional de check nhanh quality sau pipeline.


In [ ]:
from ultralytics import YOLO

def eval_map(weight_path, label):
    model = YOLO(str(weight_path))
    metrics = model.val(
        data=str(DATA_YAML_PATH),
        split="test",
        imgsz=IMGSZ,
        batch=16,
        device=DEVICE,
        workers=TRAIN_WORKERS,
        verbose=False,
    )
    return {
        "label": label,
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
    }

eval_targets = [
    (PRETRAINED_WEIGHT, "FP32 original"),
    (FINETUNE_WEIGHT, "Pruned + Finetune"),
    (QAT_WEIGHT, "Pruned + QAT"),
]

if ENGINE_PATH is not None and Path(ENGINE_PATH).exists():
    eval_targets.append((ENGINE_PATH, "TensorRT INT8 engine"))

rows = []
for w, label in eval_targets:
    try:
        row = eval_map(w, label)
        rows.append(row)
        print(f"{label:<24} mAP50={row['mAP50']:.4f}  mAP50-95={row['mAP50-95']:.4f}")
    except Exception as ex:
        print(f"{label:<24} eval failed: {ex}")

rows


In [ ]:
import cv2
import matplotlib.pyplot as plt

if ENGINE_PATH is not None and Path(ENGINE_PATH).exists():
    preview_weight = ENGINE_PATH
    preview_label = "TensorRT INT8"
else:
    preview_weight = QAT_WEIGHT
    preview_label = "QAT PyTorch"

model = YOLO(str(preview_weight))

test_dir = DATASET_LOCAL / "test" / "images"
if not test_dir.exists() or len(list(test_dir.glob("*"))) == 0:
    test_dir = DATASET_LOCAL / "val" / "images"

image_paths = sorted(test_dir.glob("*"))[:6]
assert image_paths, f"No images found in {test_dir}"

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, img_path in zip(axes.flat, image_paths):
    result = model.predict(str(img_path), conf=0.25, verbose=False)[0]
    annotated = result.plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name)
    ax.axis("off")

plt.suptitle(f"Preview - {preview_label}", fontsize=14)
plt.tight_layout()
plt.show()
